# Interactive Table Latency Benchmark

Run inside **Snowsight** (container runtime) to measure true execution latency with zero network overhead.
Disables result cache to ensure fresh execution each time.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE ECOM_STREAMING;
USE SCHEMA CLICKSTREAM;
USE WAREHOUSE ECOM_IWH;
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

## Point Lookups

In [ ]:
-- Point lookup: single user
SELECT * FROM CLICK_EVENTS WHERE USER_ID = 'user_0001' LIMIT 1;

In [ ]:
-- Point lookup: user + event type filter
SELECT * FROM CLICK_EVENTS WHERE USER_ID = 'user_0050' AND EVENT_TYPE = 'purchase' LIMIT 1;

In [ ]:
-- Latest event (ORDER BY + LIMIT 1)
SELECT * FROM CLICK_EVENTS ORDER BY EVENT_TIMESTAMP DESC LIMIT 1;

## Time-Windowed Aggregations

In [ ]:
-- Count events in last 10 seconds
SELECT COUNT(*) AS EVENT_COUNT
FROM CLICK_EVENTS
WHERE EVENT_TIMESTAMP >= DATEADD(second, -10, CURRENT_TIMESTAMP());

In [ ]:
-- Distinct active users (30s window)
SELECT COUNT(DISTINCT USER_ID) AS ACTIVE_USERS
FROM CLICK_EVENTS
WHERE EVENT_TIMESTAMP >= DATEADD(second, -30, CURRENT_TIMESTAMP());

In [ ]:
-- Group by device type (1 min)
SELECT DEVICE_TYPE, COUNT(*) AS EVENT_COUNT
FROM CLICK_EVENTS
WHERE EVENT_TIMESTAMP >= DATEADD(minute, -1, CURRENT_TIMESTAMP())
GROUP BY DEVICE_TYPE;

In [ ]:
-- Revenue in last 1 minute
SELECT COALESCE(SUM(PRODUCT_PRICE * QUANTITY), 0) AS REVENUE
FROM CLICK_EVENTS
WHERE EVENT_TYPE = 'purchase'
  AND EVENT_TIMESTAMP >= DATEADD(minute, -1, CURRENT_TIMESTAMP());

In [ ]:
-- Top product by views (1 min)
SELECT PRODUCT_NAME, COUNT(*) AS VIEW_COUNT
FROM CLICK_EVENTS
WHERE EVENT_TIMESTAMP >= DATEADD(minute, -1, CURRENT_TIMESTAMP())
  AND PRODUCT_NAME != ''
GROUP BY PRODUCT_NAME
ORDER BY VIEW_COUNT DESC
LIMIT 1;

## Dashboard-Style Queries

In [ ]:
-- Full metrics (same as dashboard)
SELECT
  COUNT(DISTINCT CASE WHEN EVENT_TIMESTAMP >= DATEADD(second, -60, CURRENT_TIMESTAMP()) THEN USER_ID END) AS ACTIVE_USERS,
  COUNT(CASE WHEN EVENT_TIMESTAMP >= DATEADD(second, -30, CURRENT_TIMESTAMP()) THEN 1 END) / 30.0 AS EVENTS_PER_SEC,
  COALESCE(SUM(CASE WHEN EVENT_TYPE = 'purchase' AND EVENT_TIMESTAMP >= DATEADD(minute, -5, CURRENT_TIMESTAMP()) THEN PRODUCT_PRICE * QUANTITY END), 0) AS REVENUE_5MIN
FROM CLICK_EVENTS;

In [ ]:
-- Conversion funnel
SELECT EVENT_TYPE, COUNT(*) AS EVENTS, COUNT(DISTINCT USER_ID) AS USERS
FROM CLICK_EVENTS
WHERE EVENT_TIMESTAMP >= DATEADD(minute, -5, CURRENT_TIMESTAMP())
  AND EVENT_TYPE IN ('page_view', 'product_view', 'add_to_cart', 'checkout', 'purchase')
GROUP BY EVENT_TYPE;

In [ ]:
-- Category performance
SELECT PRODUCT_CATEGORY,
  COUNT(CASE WHEN EVENT_TYPE = 'product_view' THEN 1 END) AS VIEWS,
  COUNT(CASE WHEN EVENT_TYPE = 'purchase' THEN 1 END) AS PURCHASES,
  COALESCE(SUM(CASE WHEN EVENT_TYPE = 'purchase' THEN PRODUCT_PRICE * QUANTITY END), 0) AS REVENUE
FROM CLICK_EVENTS
WHERE EVENT_TIMESTAMP >= DATEADD(minute, -5, CURRENT_TIMESTAMP())
  AND PRODUCT_CATEGORY != ''
GROUP BY PRODUCT_CATEGORY
ORDER BY REVENUE DESC;

## Latency Report
Actual execution times from QUERY_HISTORY for all queries in this session:

In [ ]:
-- Per-query latency breakdown
SELECT
  ROW_NUMBER() OVER (ORDER BY START_TIME) AS "#",
  SUBSTR(QUERY_TEXT, 1, 55) AS QUERY_PREVIEW,
  EXECUTION_TIME AS EXEC_MS,
  COMPILATION_TIME AS COMPILE_MS,
  TOTAL_ELAPSED_TIME AS TOTAL_MS,
  CASE
    WHEN EXECUTION_TIME <= 10 THEN 'SUB-10ms'
    WHEN EXECUTION_TIME <= 50 THEN 'SUB-50ms'
    WHEN EXECUTION_TIME <= 100 THEN 'SUB-100ms'
    ELSE CONCAT(EXECUTION_TIME::VARCHAR, 'ms')
  END AS SPEED_TIER
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 30))
WHERE WAREHOUSE_NAME = 'ECOM_IWH'
  AND QUERY_TYPE = 'SELECT'
  AND EXECUTION_STATUS = 'SUCCESS'
  AND QUERY_TEXT NOT ILIKE '%QUERY_HISTORY%'
  AND QUERY_TEXT NOT ILIKE '%USE_%'
ORDER BY START_TIME;

In [ ]:
-- Summary: aggregate latency stats
SELECT
  COUNT(*) AS TOTAL_QUERIES,
  ROUND(AVG(EXECUTION_TIME)) AS AVG_EXEC_MS,
  MIN(EXECUTION_TIME) AS MIN_EXEC_MS,
  MAX(EXECUTION_TIME) AS MAX_EXEC_MS,
  ROUND(AVG(COMPILATION_TIME)) AS AVG_COMPILE_MS,
  ROUND(AVG(TOTAL_ELAPSED_TIME)) AS AVG_TOTAL_MS,
  COUNT(CASE WHEN EXECUTION_TIME <= 10 THEN 1 END) AS UNDER_10MS,
  COUNT(CASE WHEN EXECUTION_TIME <= 50 THEN 1 END) AS UNDER_50MS,
  COUNT(CASE WHEN EXECUTION_TIME <= 100 THEN 1 END) AS UNDER_100MS
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 30))
WHERE WAREHOUSE_NAME = 'ECOM_IWH'
  AND QUERY_TYPE = 'SELECT'
  AND EXECUTION_STATUS = 'SUCCESS'
  AND QUERY_TEXT NOT ILIKE '%QUERY_HISTORY%'
  AND QUERY_TEXT NOT ILIKE '%USE_%';